# 02 — Entrainement des modeles

6 modeles compares : Regression logistique, KNN, Arbre de decision, Random Forest, SVM, XGBoost.

Donnees d'entree : `data/client_churn_dataset.csv`, produit par `01_preparation.ipynb`
via requete SQL directe sur l'entrepot PostgreSQL `churn_dw`.

**Variables exclues (fuite de donnees)** : `nb_comptes_clos` (definit litteralement
la cible) et `nb_devises` (artefact de completude des donnees produit). Voir
`comparison.md` pour le detail de l'investigation.

**Note performance** : KNN et SVM sont entraines sur un sous-echantillon (40 000 /
8 000 lignes) car ils ne passent pas a l'echelle sur ~290k lignes d'entrainement.

In [1]:
import sys
sys.path.append('../src')
from train import get_models, train_one, finalize

list(get_models().keys())

['Regression_logistique',
 'KNN',
 'Arbre_de_decision',
 'Random_Forest',
 'SVM',
 'XGBoost']

## Entrainement de chaque modele

Chaque appel entraine un modele, l'evalue sur le jeu de test, et sauvegarde ses metriques dans `data/results.json`.

In [2]:
for name in get_models():
    train_one(name)

Regression_logistique {'precision': 0.9159, 'recall': 0.8933, 'f1': 0.9044, 'roc_auc': 0.9447, 'pr_auc': 0.9492, 'temps_entrainement_s': 7.0, 'n_entrainement': 290855, 'n_evaluation': 72714}
KNN {'precision': 0.9674, 'recall': 0.8977, 'f1': 0.9312, 'roc_auc': 0.9722, 'pr_auc': 0.9737, 'temps_entrainement_s': 5.8, 'n_entrainement': 40000, 'n_evaluation': 15000}
Arbre_de_decision {'precision': 0.9616, 'recall': 0.9155, 'f1': 0.9379, 'roc_auc': 0.9743, 'pr_auc': 0.9767, 'temps_entrainement_s': 6.1, 'n_entrainement': 290855, 'n_evaluation': 72714}
Random_Forest {'precision': 0.9891, 'recall': 0.8927, 'f1': 0.9384, 'roc_auc': 0.9771, 'pr_auc': 0.9806, 'temps_entrainement_s': 40.9, 'n_entrainement': 290855, 'n_evaluation': 72714}


c:\Users\Isra\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


SVM {'precision': 0.9837, 'recall': 0.8706, 'f1': 0.9237, 'roc_auc': 0.9552, 'pr_auc': 0.9484, 'temps_entrainement_s': 31.3, 'n_entrainement': 8000, 'n_evaluation': 72714}
XGBoost {'precision': 0.9801, 'recall': 0.9076, 'f1': 0.9425, 'roc_auc': 0.9806, 'pr_auc': 0.9829, 'temps_entrainement_s': 12.9, 'n_entrainement': 290855, 'n_evaluation': 72714}


## Sélection du modèle final (meilleur PR-AUC)

Le **PR-AUC** (*Area Under the Precision-Recall Curve*) est une métrique d’évaluation utilisée pour les modèles de **classification binaire**.  
Elle mesure la performance du modèle en tenant compte de l’équilibre entre **précision** (proportion de prédictions positives correctes) et **rappel** (proportion de vrais positifs détectés).  
Un PR-AUC plus élevé indique un modèle plus performant dans la détection des classes positives, en particulier lorsque les données sont déséquilibrées.


In [3]:
results_df, best_name = finalize()
print('Modele retenu :', best_name)
results_df

Meilleur modèle (PR-AUC): XGBoost
Modele retenu : XGBoost


,precision,recall,f1,roc_auc,pr_auc,temps_entrainement_s,n_entrainement,n_evaluation
Regression_logistique,0.9159,0.8933,0.9044,0.9447,0.9492,7.0,290855.0,72714.0
KNN,0.9674,0.8977,0.9312,0.9722,0.9737,5.8,40000.0,15000.0
Arbre_de_decision,0.9616,0.9155,0.9379,0.9743,0.9767,6.1,290855.0,72714.0
Random_Forest,0.9891,0.8927,0.9384,0.9771,0.9806,40.9,290855.0,72714.0
SVM,0.9837,0.8706,0.9237,0.9552,0.9484,31.3,8000.0,72714.0
XGBoost,0.9801,0.9076,0.9425,0.9806,0.9829,12.9,290855.0,72714.0
